In [1]:
import pandas as pd
import numpy as np

In [ ]:
# load data
panel = pd.read_csv("panel_dataset.csv")
panel.head()

,name,country,region,lat,lon,bleaching_freq_significant,bleaching_freq_severe,sst_trend_per_decade
0,Great Barrier Reef - Osprey Reef,Australia,Indo-Pacific,-13.8833,146.5667,10.0,6.0,0.294188
1,Great Barrier Reef - Cod Hole,Australia,Indo-Pacific,-16.1167,145.9833,7.0,2.0,0.255187
2,Coral Sea - Holmes Reef,Australia,Indo-Pacific,-16.4833,147.8667,10.0,4.0,0.288779
3,Ribbon Reefs - No. 10,Australia,Indo-Pacific,-15.0833,145.7500,5.0,3.0,0.262589
4,Raja Ampat - Misool,Indonesia,Indo-Pacific,-2.0833,130.1667,0.0,0.0,0.193966


In [ ]:
# normalization function (to 0-100)
def normalize(series):
    return (series - series.min()) / (series.max() - series.min()) * 100

panel["score_bleaching"] = normalize(panel["bleaching_freq_significant"])
panel["score_trend"]     = normalize(panel["sst_trend_per_decade"])

print(panel[["name", "score_bleaching", "score_trend"]].head(10))

                               name  score_bleaching  score_trend
0  Great Barrier Reef - Osprey Reef             50.0    56.504320
1     Great Barrier Reef - Cod Hole             35.0    47.361285
2           Coral Sea - Holmes Reef             50.0    55.236211
3             Ribbon Reefs - No. 10             25.0    49.096538
4               Raja Ampat - Misool              0.0    33.009020
5             Raja Ampat - Cape Kri             30.0    54.780538
6              Komodo - Batu Bolong             50.0    46.016184
7             Komodo - Crystal Rock             40.0    45.118375
8            Banda Sea - Gunung Api             30.0    39.278249
9           Tulamben - USAT Liberty             35.0    40.765531


In [ ]:
# composite score and category

# Weighted score: 60% Bleaching Frequency, 40% SST Trend
# Bleaching Frequency is the more direct Indicator for coral damage; SST trend the future oriented Indicator.
W_BLEACHING = 0.6
W_TREND     = 0.4

panel["risk_score"] = (
    W_BLEACHING * panel["score_bleaching"] +
    W_TREND     * panel["score_trend"]
).round(1)

# Kategorisierung
def categorize(score):
    if pd.isna(score):   return "Unknown"
    elif score < 25:     return "Low"
    elif score < 50:     return "Medium"
    elif score < 75:     return "High"
    else:                return "Critical"

panel["risk_level"] = panel["risk_score"].apply(categorize)

print(panel["risk_level"].value_counts())

risk_level
Medium      22
Low         16
High         9
Critical     2
Unknown      1
Name: count, dtype: int64


In [5]:
# Top 10 endangered spots
top10 = panel.dropna(subset=["risk_score"]).sort_values("risk_score", ascending=False).head(10)
print(top10[["name", "country", "risk_score", "risk_level"]])

                                name               country  risk_score  \
37         Galapagos - Darwin Island               Ecuador        79.3   
30            Red Sea - Ras Mohammed                 Egypt        79.0   
40           Bonaire - Klein Bonaire  Netherlands Antilles        69.9   
31         Red Sea - Brother Islands                 Egypt        63.3   
38   Belize Barrier Reef - Blue Hole                Belize        61.9   
36           Galapagos - Wolf Island               Ecuador        58.6   
0   Great Barrier Reef - Osprey Reef             Australia        52.6   
46      Azores - Princess Alice Bank              Portugal        52.4   
47   Faial - Banco D. Joao de Castro              Portugal        52.4   
2            Coral Sea - Holmes Reef             Australia        52.1   

   risk_level  
37   Critical  
30   Critical  
40       High  
31       High  
38       High  
36       High  
0        High  
46       High  
47       High  
2        High  


In [6]:
panel.to_csv("panel_dataset.csv", index=False)
print("panel_dataset.csv mit Risk Score gespeichert")
panel.describe()

panel_dataset.csv mit Risk Score gespeichert


,lat,lon,bleaching_freq_significant,bleaching_freq_severe,sst_trend_per_decade,score_bleaching,score_trend,risk_score
count,50.000000,50.000000,50.000000,50.000000,49.000000,50.000000,49.000000,49.000000
mean,4.883336,52.720666,6.100000,2.220000,0.227797,30.500000,40.940040,35.044898
std,15.980240,90.161552,4.362409,2.349902,0.081625,21.812045,19.135559,18.432976
min,-31.550000,-156.483300,0.000000,0.000000,0.053161,0.000000,0.000000,6.000000
25%,-5.733350,-10.454150,3.000000,1.000000,0.183649,15.000000,30.590509,19.300000
50%,5.341650,85.583300,5.000000,2.000000,0.220908,25.000000,39.325216,36.500000
75%,16.020850,124.050000,8.750000,3.000000,0.286835,43.750000,54.780538,44.200000
max,38.233300,159.083300,20.000000,12.000000,0.479725,100.000000,100.000000,79.300000
